# 01 · Conexión y extracción de prueba

In [1]:
"""
Aquí lo primero que hicimos fue leer la configuración y el manifiesto desde src/, o sea que el notebook no repite lógica
del pipeline sino que la importa. Lo hicimos así porque si dejábamos las URL escritas a mano en el notebook, digamos, cualquier
cambio en config.yaml nos habría dejado dos versiones distintas de la verdad. La salida muestra las fuentes que quedaron como
primarias después de la verificación previa de las fuentes y el plan elegido. Lo que esto NO demuestra es que las fuentes sigan vivas hoy, eso lo comprobamos
en las celdas siguientes con peticiones reales.
"""
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import requests
from src.common import load_config, user_agent, read_manifest
from src.extract.extract_api import http_get

cfg = load_config()
print('plan:', cfg['plan'], '| timeout_s:', cfg['http']['timeout_s'], '| reintentos:', cfg['http']['retries'])
for k in ('edificaciones', 'estrato_barrio', 'poligonos_barrios'):
    print(k, '->', str(cfg['sources'][k].get('layer') or cfg['sources'][k]['url'])[:110])


plan: A | timeout_s: 30 | reintentos: 3
edificaciones -> https://services3.arcgis.com/Hf3YbZrlZ0OA2COM/arcgis/rest/services/Impacto_Satelital_en_Edificaciones_por_Terr
estrato_barrio -> https://datos.cali.gov.co/dataset/c6421d6b-2f7c-4a28-b126-fb60192e6302/resource/f492c39c-d980-4a97-9eaa-f00dc1
poligonos_barrios -> https://ws-idesc.cali.gov.co/geoserver/dapm/ows


In [2]:
"""
Aquí hicimos una conexión de prueba pidiendo solamente 5 filas de la capa de edificaciones. Metimos la sesión dentro de un try
porque conectarse a un servicio externo depende de cosas que no controlamos, como la red o que el servidor esté arriba, y por eso
cerramos la sesión en el finally, para no dejar sockets abiertos si algo falla a mitad de camino. Le pusimos timeout de 30 segundos
(el de config.yaml) porque sin timeout, si el servidor no contesta, el script se queda colgado para siempre. La salida muestra
las columnas reales de la capa y que NO hay nombre de barrio, solo códigos; eso no demuestra todavía que el JOIN funcione.
"""
import pandas as pd
url = cfg['sources']['edificaciones']['url']
params = {'where': '1=1', 'outFields': '*', 'returnGeometry': 'false', 'orderByFields': 'OBJECTID',
          'resultRecordCount': 5, 'f': 'json'}
session = None
try:
    session = requests.Session()
    session.headers['User-Agent'] = user_agent()
    r = http_get(session, url, params, cfg)
    print('HTTP', r.status_code, '| bytes:', len(r.content))
    prueba = pd.DataFrame([f['attributes'] for f in r.json()['features']])
finally:
    if session is not None:
        session.close()
print(prueba[['OBJECTID', 'COMUNA', 'BARRIO', 'NPISOS', 'Indice_Escombros', 'Estado_Estructural']])


HTTP 200 | bytes: 5271
   OBJECTID COMUNA BARRIO  NPISOS  Indice_Escombros  \
0         1     19     04       1         -8.341564   
1         2     03     05       3         -4.004968   
2         3     03     06       0         -3.197851   
3         4     17     85       1         -2.644952   
4         5     19     10       2         -8.608572   

                   Estado_Estructural  
0  Sin cambio aparente / Daño interno  
1  Sin cambio aparente / Daño interno  
2  Sin cambio aparente / Daño interno  
3  Sin cambio aparente / Daño interno  
4  Sin cambio aparente / Daño interno  


In [3]:
"""
Acá quisimos ver qué pasa cuando las cosas salen mal, porque un pipeline que solo funciona en el día feliz no nos sirve.
Simulamos tres fallos: una capa que no existe, una URL que da 404 y un timeout absurdo de 1 milisegundo. Nos dimos cuenta de
algo importante, y es que ArcGIS puede contestar HTTP 200 con un JSON que trae un campo error adentro, por eso en extract_api.py
miramos el cuerpo y no solo el código de estado. La salida muestra cada caso. Lo que NO prueba es el comportamiento ante cortes
largos de red, para eso solo tenemos los reintentos simples de la configuración.
"""
base = cfg['sources']['edificaciones']['layer'].rsplit('/', 1)[0]
s = requests.Session()
try:
    r = s.get(base + '/99/query', params={'where': '1=1', 'f': 'json'}, timeout=30)
    print('capa inexistente -> HTTP', r.status_code, '| cuerpo:', r.text[:140].replace('\n', ' '))
    r = s.get('https://datos.cali.gov.co/dataset/no-existe-xyz', timeout=30, headers={'User-Agent': user_agent()})
    print('URL inexistente  -> HTTP', r.status_code)
    try:
        s.get(url, params=params, timeout=0.001)
    except requests.RequestException as e:
        print('timeout 1 ms     ->', type(e).__name__)
finally:
    s.close()


capa inexistente -> HTTP 200 | cuerpo: {"error":{"code":400,"message":"Invalid URL","details":["Invalid URL"]}}


URL inexistente  -> HTTP 404
timeout 1 ms     -> ReadTimeout


In [4]:
"""
En esta celda corrimos la extracción real llamando a las mismas funciones que usa main.py, así que el notebook y la línea de
comandos no pueden divergir. Como Bronze ya existía desde la primera corrida, lo esperado es ver los mensajes de que el archivo ya
existe y el sha256 coincide, y que no se crea nada nuevo; eso es la idempotencia que exigimos como regla del proyecto. La salida confirma esa
conducta, pero no demuestra que los datos de la fuente no hayan cambiado desde que los bajamos, eso solo lo diría un refresco
con --refresh que compare sha256.
"""
from main import run_extract
n_antes = len(read_manifest())
run_extract(refresh=False)
print('líneas del manifiesto antes/después:', n_antes, len(read_manifest()))


2026-10-16 14:47:26 INFO etl.main Inicio etapa extract


2026-10-16 14:47:26 INFO etl.extract_api edificaciones: ya existe, sha256 coincide (3 páginas)


2026-10-16 14:47:26 INFO etl.extract_excel estrato_barrio: ya existe, sha256 coincide


2026-10-16 14:47:26 INFO etl.extract_api poligonos_barrios: ya existe, sha256 coincide


2026-10-16 14:47:26 INFO etl.main Fin etapa extract (0.07 s)


líneas del manifiesto antes/después: 8 8


In [5]:
"""
Aquí armamos una tabla con el manifiesto para tener a la vista qué quedó guardado: una línea por archivo, con filas, bytes y sha256.
Lo hicimos porque el manifiesto es nuestra evidencia de trazabilidad, y es más fácil verlo en tabla que en JSON
crudo. Fíjese, digamos, que hay dos conteos (antes y después de descargar) y que ambos dan 5.146, igual que la suma de las tres
páginas (2.000 + 2.000 + 1.146). Eso muestra que la capa no cambió durante la descarga; lo que no muestra es que la capa esté
completa respecto a la realidad, porque la fuente solo cubre edificaciones dentro de los polígonos de afectación preliminar.
"""
man = pd.DataFrame(read_manifest())
pd.set_option('display.width', 200, 'display.max_colwidth', 70)
print(man[['source_id', 'http_status', 'rows', 'bytes', 'sha256', 'path']].assign(sha256=lambda d: d.sha256.str[:12] + '…'))
paginas = man[man.source_id == 'edificaciones']
print('filas por página:', paginas.rows.tolist(), '| suma:', paginas.rows.sum())


                     source_id  http_status  rows    bytes         sha256                                                                   path
0    edificaciones_count_antes          200  5146      181  c20dec67463c…   data/bronze/api/edificaciones/snap_20261016T001024Z/count_antes.json
1      edificaciones_capa_meta          200    20    17507  c5e4f4258775…     data/bronze/api/edificaciones/snap_20261016T001024Z/capa_meta.json
2                edificaciones          200  2000  1774634  92dfda868878…     data/bronze/api/edificaciones/snap_20261016T001024Z/page_0001.json
3                edificaciones          200  2000  1752478  6b2a4849e95d…     data/bronze/api/edificaciones/snap_20261016T001024Z/page_0002.json
4                edificaciones          200  1146  1055160  ac23590d1a67…     data/bronze/api/edificaciones/snap_20261016T001024Z/page_0003.json
5  edificaciones_count_despues          200  5146      181  c20dec67463c…  data/bronze/api/edificaciones/snap_20261016T001024Z/cou

In [6]:
"""
Por último verificamos de forma independiente que cada archivo de Bronze tenga el sha256 que dice el manifiesto, recalculándolo
nosotros desde el disco. Lo hicimos porque Bronze es inmutable y la única manera de saber que nadie lo tocó es comparar hashes.
Además comparamos la página 1 contra una petición nueva a la API, byte a byte. La salida debería mostrar True en todo; si la
fuente hubiera cambiado entre corridas veríamos False en la comparación con la API, y eso sería un cambio real de la fuente, no un
error nuestro. Esto no prueba que la fuente sea correcta, solo que la copia local es fiel a lo que nos respondió.
"""
from src.common import sha256_file, ROOT
ok = all(sha256_file(ROOT / m['path']) == m['sha256'] for m in read_manifest())
print('todos los sha256 coinciden con el manifiesto:', ok)
p1 = next(m for m in read_manifest() if m['source_id'] == 'edificaciones')
s = requests.Session()
try:
    nueva = http_get(s, p1['url'], p1['params'], cfg)
finally:
    s.close()
print('página 1 idéntica a una petición nueva:', nueva.content == (ROOT / p1['path']).read_bytes())


todos los sha256 coinciden con el manifiesto: True


página 1 idéntica a una petición nueva: True
